# Comparación de prompts para optimizar costo con tu modelo fine-tuned

Esta notebook prueba varias versiones de prompt sobre el mismo conjunto de casos y compara etiquetas, tokens y costo estimado.

In [ ]:
# Si hace falta:
# !pip install -q openai pandas tqdm

In [ ]:
import os
import json
import time
from typing import Any, Dict, Optional

import pandas as pd
from tqdm.auto import tqdm
from openai import OpenAI

In [ ]:
MODEL_NAME = 'ft:gpt-3.5-turbo-0125:personal:hatemultiv4:9uouAqOs'
INPUT_CSV = 'muestra_100.csv'
OUTPUT_CSV = 'muestra_100_prompt_benchmark.csv'

TEXT_COLUMN = None
MAX_ROWS = 100

TEMPERATURE = 0
MAX_TOKENS = 40
SLEEP_SECONDS = 0.1

# Ajusta estos valores si quieres usar otra tarifa estimada
PRICE_INPUT_PER_1M = 3.00
PRICE_OUTPUT_PER_1M = 6.00

client = OpenAI(api_key=os.environ.get('OPENAI_API_KEY'))
if not os.environ.get('OPENAI_API_KEY'):
    raise ValueError('No encuentro OPENAI_API_KEY en variables de entorno.')

## Carga del dataset

In [ ]:
df = pd.read_csv(INPUT_CSV)
print(f'Filas totales: {len(df)}')
print('Columnas:')
print(list(df.columns))
df.head(3)

In [ ]:
def detect_text_column(dataframe: pd.DataFrame) -> str:
    candidates = ['text', 'texto', 'tweet', 'contenido', 'content', 'body', 'comment', 'comentario', 'post', 'mensaje', 'message']
    lower_map = {c.lower(): c for c in dataframe.columns}
    for cand in candidates:
        if cand in lower_map:
            return lower_map[cand]
    object_cols = [c for c in dataframe.columns if dataframe[c].dtype == 'object']
    if len(object_cols) == 1:
        return object_cols[0]
    raise ValueError('No pude detectar automáticamente la columna de texto. Define TEXT_COLUMN manualmente.')

text_col = TEXT_COLUMN if TEXT_COLUMN is not None else detect_text_column(df)
work_df = df.copy().head(MAX_ROWS).reset_index(drop=True)
print('Columna de texto:', text_col)
print('Filas a procesar:', len(work_df))

## Variantes de prompt

In [ ]:
PROMPTS = {
    'baseline_largo': {
        'system': 'Eres un clasificador multietiqueta de discursos de odio. Responde únicamente con JSON válido en una sola línea.',
        'user_template': 'Clasifica el siguiente texto usando tu esquema de etiquetas.\n\nDevuelve SOLO JSON válido con esta estructura:\n{"labels": ["RACISM"]}\n\nSi no corresponde ninguna etiqueta, usa una lista vacía. No agregues explicación fuera del JSON.\n\nTexto:\n{text}'
    },
    'medio': {
        'system': 'Devuelve solo JSON válido.',
        'user_template': 'Clasifica el texto y responde solo con JSON:\n{"labels": []}\n\nTexto:\n{text}'
    },
    'minimo': {
        'system': '',
        'user_template': '{"labels": []}\nTexto:\n{text}'
    }
}

## Funciones auxiliares

In [ ]:
def safe_json_loads(text: str) -> Optional[Dict[str, Any]]:
    if text is None:
        return None
    text = str(text).strip()
    try:
        return json.loads(text)
    except Exception:
        pass
    start = text.find('{')
    end = text.rfind('}')
    if start != -1 and end != -1 and end > start:
        candidate = text[start:end+1]
        try:
            return json.loads(candidate)
        except Exception:
            return None
    return None

def normalize_labels(parsed_json: Optional[Dict[str, Any]]) -> str:
    if not isinstance(parsed_json, dict):
        return 'PARSE_ERROR'
    labels = parsed_json.get('labels')
    if labels is None:
        return 'MISSING_LABELS'
    if isinstance(labels, list):
        labels = [str(x).strip() for x in labels]
        labels = sorted([x for x in labels if x])
        return '|'.join(labels) if labels else 'NONE'
    return str(labels)

def call_model(text: str, prompt_variant: Dict[str, str]) -> Dict[str, Any]:
    messages = []
    if prompt_variant['system'].strip():
        messages.append({'role': 'system', 'content': prompt_variant['system']})
    messages.append({'role': 'user', 'content': prompt_variant['user_template'].format(text=text)})

    response = client.chat.completions.create(
        model=MODEL_NAME,
        temperature=TEMPERATURE,
        max_tokens=MAX_TOKENS,
        messages=messages,
    )

    raw = response.choices[0].message.content
    parsed = safe_json_loads(raw)
    usage = getattr(response, 'usage', None)

    return {
        'raw_response': raw,
        'parsed_json': parsed,
        'labels_norm': normalize_labels(parsed),
        'prompt_tokens': getattr(usage, 'prompt_tokens', None),
        'completion_tokens': getattr(usage, 'completion_tokens', None),
        'total_tokens': getattr(usage, 'total_tokens', None),
        'finish_reason': response.choices[0].finish_reason,
        'model_used': response.model,
    }

## Prueba con una fila

In [ ]:
sample_text = str(work_df[text_col].iloc[0])
sample_results = {}
for prompt_name, prompt_variant in PROMPTS.items():
    sample_results[prompt_name] = call_model(sample_text, prompt_variant)
sample_results

## Corrida completa

In [ ]:
results = []

for idx, row in tqdm(work_df.iterrows(), total=len(work_df)):
    text = str(row[text_col]) if pd.notna(row[text_col]) else ''
    row_result = {'row_id': idx, 'text': text}

    for prompt_name, prompt_variant in PROMPTS.items():
        try:
            out = call_model(text, prompt_variant)
            row_result[f'{prompt_name}__raw_response'] = out['raw_response']
            row_result[f'{prompt_name}__labels_norm'] = out['labels_norm']
            row_result[f'{prompt_name}__prompt_tokens'] = out['prompt_tokens']
            row_result[f'{prompt_name}__completion_tokens'] = out['completion_tokens']
            row_result[f'{prompt_name}__total_tokens'] = out['total_tokens']
            row_result[f'{prompt_name}__finish_reason'] = out['finish_reason']
            row_result[f'{prompt_name}__error'] = None
        except Exception as e:
            row_result[f'{prompt_name}__raw_response'] = None
            row_result[f'{prompt_name}__labels_norm'] = None
            row_result[f'{prompt_name}__prompt_tokens'] = None
            row_result[f'{prompt_name}__completion_tokens'] = None
            row_result[f'{prompt_name}__total_tokens'] = None
            row_result[f'{prompt_name}__finish_reason'] = None
            row_result[f'{prompt_name}__error'] = str(e)

        time.sleep(SLEEP_SECONDS)

    results.append(row_result)

res_df = pd.DataFrame(results)
res_df.to_csv(OUTPUT_CSV, index=False)
print(f'Resultados guardados en: {OUTPUT_CSV}')

## Resumen de tokens y costo

In [ ]:
summary_rows = []

for prompt_name in PROMPTS.keys():
    p_in = pd.to_numeric(res_df[f'{prompt_name}__prompt_tokens'], errors='coerce').fillna(0)
    p_out = pd.to_numeric(res_df[f'{prompt_name}__completion_tokens'], errors='coerce').fillna(0)
    p_tot = pd.to_numeric(res_df[f'{prompt_name}__total_tokens'], errors='coerce').fillna(0)

    input_cost = p_in.sum() / 1_000_000 * PRICE_INPUT_PER_1M
    output_cost = p_out.sum() / 1_000_000 * PRICE_OUTPUT_PER_1M
    total_cost = input_cost + output_cost

    summary_rows.append({
        'prompt': prompt_name,
        'rows': len(res_df),
        'avg_prompt_tokens': round(p_in.mean(), 2),
        'avg_completion_tokens': round(p_out.mean(), 2),
        'avg_total_tokens': round(p_tot.mean(), 2),
        'sum_prompt_tokens': int(p_in.sum()),
        'sum_completion_tokens': int(p_out.sum()),
        'sum_total_tokens': int(p_tot.sum()),
        'estimated_cost_sample': round(total_cost, 6),
        'estimated_cost_250k': round(total_cost * (250000 / len(res_df)), 2)
    })

summary_df = pd.DataFrame(summary_rows).sort_values('estimated_cost_250k')
summary_df

## Acuerdo entre prompts

In [ ]:
def agreement_rate(col_a: str, col_b: str) -> float:
    tmp = res_df[[col_a, col_b]].dropna()
    if len(tmp) == 0:
        return float('nan')
    return (tmp[col_a] == tmp[col_b]).mean()

pairs = []
prompt_names = list(PROMPTS.keys())
for i in range(len(prompt_names)):
    for j in range(i + 1, len(prompt_names)):
        a = prompt_names[i]
        b = prompt_names[j]
        pairs.append({
            'prompt_a': a,
            'prompt_b': b,
            'agreement': round(agreement_rate(f'{a}__labels_norm', f'{b}__labels_norm'), 4)
        })

agreement_df = pd.DataFrame(pairs).sort_values('agreement', ascending=False)
agreement_df

## Casos donde difieren

In [ ]:
comparison_cols = ['text'] + [f'{p}__labels_norm' for p in PROMPTS.keys()]
diff_df = res_df[comparison_cols].copy()
diff_df['n_unique_outputs'] = diff_df[[f'{p}__labels_norm' for p in PROMPTS.keys()]].nunique(axis=1)
diff_df[diff_df['n_unique_outputs'] > 1].head(20)

## Recomendación automática simple

In [ ]:
baseline = 'baseline_largo'
recommendations = []
for prompt_name in PROMPTS.keys():
    if prompt_name == baseline:
        continue
    agr = agreement_rate(f'{baseline}__labels_norm', f'{prompt_name}__labels_norm')
    row = summary_df[summary_df['prompt'] == prompt_name].iloc[0].to_dict()
    row['agreement_vs_baseline'] = round(agr, 4)
    recommendations.append(row)

rec_df = pd.DataFrame(recommendations).sort_values(['agreement_vs_baseline', 'estimated_cost_250k'], ascending=[False, True])
rec_df